In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
from pydantic import BaseModel
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM
from crewai_tools import ScrapeWebsiteTool
from crewai.flow.flow import Flow, listen, router, start

class LeadState(BaseModel):
    company_name: str = ""
    website_url: str = ""
    scraped_info: str = ""
    qualification_tier: str = ""
    action_plan: str = ""

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.3,
        max_tokens=1000,
    )

    class LeadQualificationFlow(Flow[LeadState]):
        @start()
        def capture_lead(self):
            name = input("Enter company name [or press Enter for default]: ").strip()
            if not name:
                name = "Notion"
            url = input("Enter company website URL [or press Enter for default]: ").strip()
            if not url:
                url = "https://www.notion.so"
            self.state.company_name = name
            self.state.website_url = url

        @listen(capture_lead)
        def enrich_lead(self):
            researcher = Agent(
                role="Sales Intelligence Researcher",
                goal=f"Extract value proposition and target audience for {self.state.company_name} from {self.state.website_url}",
                backstory="B2B SDR skilled in researching target accounts and tech stacks.",
                llm=llm,
                tools=[ScrapeWebsiteTool()],
                verbose=False,
            )
            task = Task(
                description=f"Inspect website {self.state.website_url} for {self.state.company_name} and summarize main products and customer profile.",
                expected_output="A concise paragraph summarizing company offerings and target market.",
                agent=researcher,
            )
            my_crew = Crew(
                agents=[researcher],
                tasks=[task],
                verbose=True,
            )
            execution_result = my_crew.kickoff()
            final_text = execution_result.raw
            self.state.scraped_info = final_text
            print(final_text)

        @listen(enrich_lead)
        def score_lead(self):
            prompt = (
                f"Classify this prospect as exactly one tier: hot, warm, or cold.\n"
                f"Company: {self.state.company_name}\n"
                f"Website Info: {self.state.scraped_info}\n"
                f"Reply with ONLY the single tier word: hot, warm, or cold."
            )
            tier = llm.call(prompt).strip().lower()
            if "hot" in tier:
                self.state.qualification_tier = "hot"
            elif "warm" in tier:
                self.state.qualification_tier = "warm"
            else:
                self.state.qualification_tier = "cold"

        @router(score_lead)
        def route_by_tier(self):
            return self.state.qualification_tier

        @listen("hot")
        def handle_hot_lead(self):
            prompt = f"Draft a personalized, high-conversion cold outreach email to {self.state.company_name} proposing an enterprise partnership based on: {self.state.scraped_info}"
            self.state.action_plan = llm.call(prompt)

        @listen("warm")
        def handle_warm_lead(self):
            self.state.action_plan = f"Enrolled {self.state.company_name} into weekly product case-study nurture campaign."

        @listen("cold")
        def handle_cold_lead(self):
            self.state.action_plan = f"Logged {self.state.company_name} in CRM archive. No immediate outbound planned."

    flow = LeadQualificationFlow()
    flow.kickoff()
    print(f"\n--- Prospect: {flow.state.company_name} | Qualification: {flow.state.qualification_tier.upper()} ---")
    print("Action Plan / Outreach:\n", flow.state.action_plan)

except Exception as e:
    print("Error:", e)
